# E-commerce Orders Query Chatbot using Agentic AI

## A teaching-grade, multi-turn case study

This lab builds and evaluates a governed support agent that remembers conversation
context, resolves pronouns such as **it**, protects cross-customer data, grounds answers
in SQLite tools, applies return and cancellation policy, escalates consequential actions,
and collects conversation-level customer feedback.

**Webinar promise:** by the end, learners can explain the architecture, inspect the state
transitions, run a multi-turn scenario, distinguish automated evaluation from customer
satisfaction, and identify what must change before production.

## 1. Why the original single-turn pattern is insufficient

A query parser that expects `customer_id` and `order_id` in every message is a useful
first exercise, but it is not a realistic support conversation. A serious agent needs:

1. **Session identity:** identity is established before chat and remains in context.
2. **Conversation memory:** an active order can be referenced later as “it.”
3. **Slot resolution:** the agent asks for only the information that is genuinely missing.
4. **Authorization:** remembered context never bypasses an ownership check.
5. **Governed tools:** the model cannot issue arbitrary database commands.
6. **Policy and approval:** a cancellation request is a proposal, not an automatic write.
7. **Evaluation:** labelled test cases and customer feedback are measured separately.

This notebook uses a package called `kartify_agent`. Packaging cuts notebook boilerplate
and makes the same tested logic reusable in Jupyter, Streamlit, and automated tests.

## 2. Environment setup

Run this notebook from the repository root. Before opening Jupyter, create a dedicated
environment and install the requirements once:

```text
python -m venv .venv
.venv\Scripts\activate
python -m pip install -r requirements.txt
python -m ipykernel install --user --name kartify-webinar --display-name "Kartify Webinar"
jupyter notebook
```

The activation command shown is for Windows. On macOS or Linux, use
`source .venv/bin/activate`. A dedicated environment prevents the dependency conflicts
that occur when a notebook upgrades packages inside a large existing Anaconda or Azure ML
environment. The notebook deliberately does not run `%pip install`, so executing it cannot
silently modify the learner's environment.

In [1]:
# EXPECTED OUTPUT: Python version, repository root, and "Dependency preflight: PASS".
# INTERPRETATION: The notebook is running from the correct folder in an environment that
# already contains the required libraries. This cell does not install or upgrade anything.
# TEACHING NOTE: Environment creation belongs before the webinar, not inside the live lab.
import importlib.util
import sys
from pathlib import Path

repository_root = Path.cwd()
required_modules = ["langgraph", "pandas", "pydantic", "rapidfuzz", "matplotlib", "IPython"]
missing_modules = [
    name for name in required_modules if importlib.util.find_spec(name) is None
]
if not (repository_root / "kartify_agent").is_dir():
    raise RuntimeError(
        "Open this notebook from the extracted webinar_kit folder so kartify_agent is visible."
    )
if missing_modules:
    raise RuntimeError(
        "Missing dependencies: " + ", ".join(missing_modules)
        + ". Activate the dedicated Kartify environment and install requirements.txt once."
    )
print("Python:", sys.version.split()[0])
print("Repository root:", repository_root)
print("Dependency preflight: PASS")

Python: 3.12.13
Repository root: /workspace/scratch/0548d158c49b/webinar_kit
Dependency preflight: PASS


In [2]:
# EXPECTED OUTPUT: A compact table with counts for four database tables.
# INTERPRETATION: The case study uses a small, auditable dataset: 5 customers, 10 orders,
# 21 order items, and 5 products. Small data is deliberate for transparent reasoning.
# TEACHING NOTE: Scale is not the learning objective; state, controls, and evaluation are.
import os
from pathlib import Path
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import SVG, Markdown, display

from kartify_agent import (
    FeedbackStore, SupportSession, benchmark_summary, build_graph, run_benchmark
)
from kartify_agent.evaluation import confusion_matrix
from kartify_agent.models import AgentState
from kartify_agent.notebook_support import show_turn
from kartify_agent.repository import database_summary, list_customer_orders, list_customers

pd.Series(database_summary(), name="rows").rename_axis("table").to_frame()

Out[2]: 
             rows
table            
customers       5
orders         10
order_items    21
products        5


### Data interpretation

Each retrieval tool is parameterized and read-only. The agent is not given a generic SQL
execution tool. This narrows the action space and makes authorization testable. In a real
application, these tools would call governed services rather than a local SQLite file.

In [3]:
# EXPECTED OUTPUT: Five demo identities followed by a newest-first order table for customer 3.
# INTERPRETATION: Customer 3 has multiple orders, so “return an order” is ambiguous. This
# creates a realistic slot-resolution test rather than allowing the agent to guess.
# TEACHING NOTE: The Streamlit selector simulates identity established outside the chat.
customers = pd.DataFrame(list_customers())
customer_3_orders = pd.DataFrame(list_customer_orders(3))
display(customers)
display(customer_3_orders)

   customer_id           name
0            1  Alice Johnson
1            2      Bob Smith
2            3  Charlie Brown
3            4   Diana Prince
4            5     Ethan Hunt

  order_id  order_date     status delivery_date  total_amount
0  ORD1008  2025-10-24    Shipped          None        299.99
1  ORD1004  2025-10-01   Returned          None         49.99
2  ORD1005  2025-09-18  Cancelled          None       1448.98

## 3. Executable agent architecture

The diagram is not merely a conceptual sequence. The same node names are compiled into a
LangGraph state machine. Controlled branches stop unsafe input, request missing context,
deny unauthorized access, or continue to tools and policy.

**Important distinction:** conversation memory helps resolve references. It does not grant
permission. Object-level authorization still runs before any private order aggregate is
loaded.

In [4]:
# EXPECTED OUTPUT: A vertical architecture diagram from UI and session context through
# guardrails, understanding, context, authorization, tools, policy, response, and evaluation.
# INTERPRETATION: Feedback is outside the turn graph because it is a customer-provided
# conversation outcome, while the evaluation node produces internal control signals.
# TEACHING NOTE: Follow the arrows once for a safe request and once for a blocked request.
display(SVG(filename="assets/agent_architecture.svg"))

<IPython.core.display.SVG object>

In [5]:
# EXPECTED OUTPUT: Lists of compiled graph node names and directed edges.
# INTERPRETATION: The visible architecture corresponds to executable routing, not a slide-only
# picture. Conditional edges are responsible for early exits and clarification paths.
# TEACHING NOTE: Ask learners which node must run before private data can enter state.
graph = build_graph()
executable = graph.get_graph()
print("Nodes:", sorted(executable.nodes))
print("Edges:")
for edge in executable.edges:
    print(f"  {edge.source} -> {edge.target}")

Nodes: ['__end__', '__start__', 'authorize', 'context', 'evaluate', 'guardrail', 'policy', 'respond', 'tools', 'understand']
Edges:
  __start__ -> guardrail
  authorize -> tools
  context -> authorize
  context -> respond
  guardrail -> respond
  guardrail -> understand
  policy -> respond
  respond -> evaluate
  tools -> policy
  understand -> context
  evaluate -> __end__


In [6]:
# EXPECTED OUTPUT: A table of the typed fields carried through one agent turn.
# INTERPRETATION: State is an explicit contract containing identity, intent, active order,
# authorization, evidence, policy, outcome, quality, and trace. This is easier to test than
# hidden prompt-only memory.
# TEACHING NOTE: Separate data fields from decisions so each claim can be traced to evidence.
state_contract = pd.DataFrame(
    {"state_field": AgentState.__annotations__.keys()}
).assign(category=lambda frame: frame.state_field.map(
    lambda name: "control" if name in {"authorized", "blocked", "handoff", "outcome", "quality"}
    else "evidence or context"
))
state_contract

Out[6]: 
                    state_field             category
0                         query  evidence or context
1                          mode  evidence or context
2                 authenticated  evidence or context
3               identity_source  evidence or context
4                   customer_id  evidence or context
5                 customer_name  evidence or context
6           claimed_customer_id  evidence or context
7               active_order_id  evidence or context
8           active_product_name  evidence or context
9               previous_intent  evidence or context
10               pending_intent  evidence or context
11  pending_candidate_order_ids  evidence or context
12                       intent  evidence or context
13                     order_id  evidence or context
14       order_reference_source  evidence or context
15             candidate_orders  evidence or context
16          candidate_order_ids  evidence or context
17          needs_clarification  evid

## 4. Multi-turn context: no repeated customer or order identifiers

The user selects a demo identity once. The first request resolves the latest order. Later
turns inherit the active order only as conversational context. Every retrieval still checks
that the session customer owns that order.

In [7]:
# EXPECTED OUTPUT: Three natural answers about ORD1003. The second and third prompts do not
# repeat customer 2 or ORD1003. The return answer ends in a human handoff.
# INTERPRETATION: “it” and “the blender” resolve through active-order memory. This is the
# core improvement over a single-turn regular-expression chatbot.
# TEACHING NOTE: Pause after each turn and ask which state fields changed.
customer_2 = SupportSession(customer_id=2)
prompts = [
    "Where is my latest order?",
    "What products are in it?",
    "Can I return the blender?",
]
results = []
for prompt in prompts:
    turn = customer_2.ask(prompt)
    results.append(turn)
    print(f"USER: {prompt}\nAGENT: {turn['response']}\n")

USER: Where is my latest order?
AGENT: ORD1003 is Processing. A delivery date has not yet been assigned; I will not invent one. Total: $449.98.

USER: What products are in it?
AGENT: ORD1003 contains:
• 4K Monitor 27" × 1 — warranty 3 years; return window 30 days
• Portable Blender × 2 — warranty 6 months; return window 14 days

USER: Can I return the blender?
AGENT: ORD1003 does not appear eligible for a return for Portable Blender. At the reproducible policy date it was 65 days old; the applicable window is 14 days. A support specialist must confirm condition and exceptions.



In [8]:
# EXPECTED OUTPUT: Three history rows. active_order_id remains ORD1003 while intent changes
# from order_status to product_help to return_help.
# INTERPRETATION: Conversation memory retains the entity, not a vague prose summary. Typed
# memory makes pronoun resolution deterministic and inspectable.
# TEACHING NOTE: A production system should set retention, deletion, and PII controls.
history = pd.DataFrame(customer_2.history)
history[["turn", "query", "intent", "active_order_id", "outcome", "quality_score", "latency_ms"]]

Out[8]: 
   turn                      query  ... quality_score latency_ms
0     1  Where is my latest order?  ...           1.0       5.16
1     2   What products are in it?  ...           1.0       3.10
2     3  Can I return the blender?  ...           1.0       3.20

[3 rows x 7 columns]


In [9]:
# EXPECTED OUTPUT: The trace and quality signals for the third turn. context should report
# conversation_memory, tools should load one order, and policy should assess a return window.
# INTERPRETATION: The response can be reconstructed from explicit decisions and governed
# evidence. Customer rating remains empty because self-evaluation is not satisfaction.
# TEACHING NOTE: Trace visibility is useful for debugging, audit, and failure attribution.
show_turn(results[-1])

<IPython.core.display.Markdown object>

         node  ...                                    data_used
0   guardrail  ...                                           []
1  understand  ...     [query, previous_intent, pending_intent]
2     context  ...        [active_order_id, scoped_order_index]
3   authorize  ...                      [customer_id, order_id]
4       tools  ...               [order, order_items, products]
5      policy  ...  [order_date, status, product.return_policy]
6     respond  ...                              [order, policy]
7    evaluate  ...    [trace, authorization, grounding, policy]

[8 rows x 5 columns]

                        value
access_control           True
grounded                 True
policy_checked           True
trace_complete           True
automated_quality_score   1.0
customer_rating          None
latency_ms                3.2

### Clarification is a continuation, not a context reset

A subtle failure occurs when the customer asks about products without identifying the
order. The agent must remember that the unfinished task is `product_help`. If the next
message is only an order number, it should complete the product request, not start a new
status request. The following six-turn scenario is the live release gate requested for
this webinar.

In [10]:
# EXPECTED OUTPUT: Six connected responses for Alice. Turn 1 asks for an order. Turn 2,
# containing only ORD1009, immediately lists Smartwatch X instead of returning order status.
# Turns 3 to 6 answer warranty, tracking, delivery, and return questions using the same order.
# INTERPRETATION: pending_intent bridges the clarification; active_order_id and
# active_product_name provide durable, typed context after the selection is resolved.
# TEACHING NOTE: Ask learners to identify when pending context is cleared and durable context begins.
clarification_session = SupportSession(customer_id=1)
clarification_prompts = [
    "Can you check and tell me which products are there in my order?",
    "ORD1009",
    "What warranty does it have?",
    "Where is it now?",
    "When will it arrive?",
    "Can I return it?",
]
clarification_results = []
for prompt in clarification_prompts:
    turn = clarification_session.ask(prompt)
    clarification_results.append(turn)
    print(f"USER: {prompt}\nAGENT: {turn['response']}\n")

USER: Can you check and tell me which products are there in my order?
AGENT: Which order should I use? Choose one of your recent orders:
• ORD1009 — Shipped — ordered 2025-08-23
• ORD1006 — Cancelled — ordered 2025-08-09

USER: ORD1009
AGENT: ORD1009 contains:
• Smartwatch X × 1 — warranty 1 year; return window 30 days

USER: What warranty does it have?
AGENT: The Smartwatch X in ORD1009 has a 1 year warranty and a 30 days return window.

USER: Where is it now?
AGENT: ORD1009 is Shipped. A delivery date has not yet been assigned; I will not invent one. Total: $299.99.

USER: When will it arrive?
AGENT: ORD1009 is Shipped. A delivery date has not yet been assigned; I will not invent one. Total: $299.99.

USER: Can I return it?
AGENT: ORD1009 does not appear eligible for a return for Smartwatch X. At the reproducible policy date it was 69 days old; the applicable window is 30 days. A support specialist must confirm condition and exceptions.



In [11]:
# EXPECTED OUTPUT: Six rows. The first row has no active order and a pending product_help
# intent. Rows 2 to 6 retain ORD1009; later intents change while the order remains stable.
# INTERPRETATION: The graph retains the minimum structured state needed for continuity.
# It does not feed an unbounded transcript back into every tool or authorization decision.
# TEACHING NOTE: Durable context must remain observable, revocable, and authorization-scoped.
clarification_history = pd.DataFrame(clarification_session.history)
clarification_history[[
    "turn", "query", "intent", "active_order_id", "active_product_name",
    "pending_intent", "outcome", "quality_score"
]]

Out[11]: 
   turn  ... quality_score
0     1  ...           1.0
1     2  ...           1.0
2     3  ...           1.0
3     4  ...           1.0
4     5  ...           1.0
5     6  ...           1.0

[6 rows x 8 columns]


## 5. Ambiguity, privacy, and action boundaries

Deep agentic behaviour is most visible at the boundaries. The next experiments show three
different non-happy paths: missing context, cross-customer access, and a consequential write.

In [12]:
# EXPECTED OUTPUT: A clarification listing customer 3's recent orders.
# INTERPRETATION: With three candidate orders and no active order, the agent asks for one
# missing slot. Guessing would create a high-risk false positive.
# TEACHING NOTE: Clarification quality is part of task success, not a chatbot failure.
ambiguous_session = SupportSession(customer_id=3)
ambiguous = ambiguous_session.ask("Can I return an order?")
print(ambiguous["response"])
pd.Series({
    "outcome": ambiguous["outcome"],
    "candidate_order_ids": ambiguous["candidate_order_ids"],
    "active_order_id": ambiguous.get("active_order_id"),
})

Which order should I use? Choose one of your recent orders:
• ORD1008 — Shipped — ordered 2025-10-24
• ORD1004 — Returned — ordered 2025-10-01
• ORD1005 — Cancelled — ordered 2025-09-18
Out[12]: 
outcome                              clarification
candidate_order_ids    [ORD1008, ORD1004, ORD1005]
active_order_id                               None
dtype: object


In [13]:
# EXPECTED OUTPUT: A privacy-preserving denial. authorized is False and order is None.
# INTERPRETATION: Customer 1 cannot retrieve ORD1001 because it belongs to customer 5. The
# private row never enters graph state, which is stronger than hiding it only in the response.
# TEACHING NOTE: Identity selection is simulated; production requires real authentication.
privacy_session = SupportSession(customer_id=1)
privacy = privacy_session.ask("Show ORD1001")
pd.Series({
    "response": privacy["response"],
    "authorized": privacy["authorized"],
    "private_order_loaded": privacy.get("order") is not None,
    "outcome": privacy["outcome"],
})

Out[13]: 
response                I cannot access that order from the active cus...
authorized                                                          False
private_order_loaded                                                False
outcome                                                            denied
dtype: object


In [14]:
# EXPECTED OUTPUT: The agent tracks ORD1002, then proposes a cancellation handoff. The
# write_executed flag remains False.
# INTERPRETATION: The agent can reason about eligibility but cannot silently mutate an order.
# A human or separately authorized workflow owns the final write.
# TEACHING NOTE: This is a practical human-in-the-loop boundary for consequential actions.
cancellation_session = SupportSession(customer_id=4)
cancellation_session.ask("Track ORD1002")
cancellation = cancellation_session.ask("Cancel it")
pd.Series({
    "response": cancellation["response"],
    "eligible_to_request": cancellation["policy"]["eligible_to_request"],
    "handoff": cancellation["handoff"],
    "write_executed": cancellation["write_executed"],
    "outcome": cancellation["outcome"],
})

Out[14]: 
response               ORD1002 is Processing, so I can prepare a canc...
eligible_to_request                                                 True
handoff                                                             True
write_executed                                                     False
outcome                                                    human_handoff
dtype: object


In [15]:
# EXPECTED OUTPUT: A safe refusal and a three-node trace: guardrail, respond, evaluate.
# INTERPRETATION: The unsafe database-write instruction exits before understanding,
# authorization, and tools. This is structural prevention, not prompt wording alone.
# TEACHING NOTE: Verify that the orders table still contains 10 rows after this experiment.
from kartify_agent import ask

blocked = ask("Drop table orders")
print(blocked["response"])
print("Trace:", [event["node"] for event in blocked["trace"]])
print("Orders after request:", database_summary()["orders"])

I can help with order information, but I cannot execute or bypass protected operations.
Trace: ['guardrail', 'respond', 'evaluate']
Orders after request: 10


## 6. Evaluation is a dataset, not a vibe

A useful release gate contains labelled multi-turn cases and safety cases. Each row below
compares expected intent, resolved order, outcome, access control, and groundedness against
the actual graph output. A passing demo is not proof of production readiness, but it is
reproducible evidence that specific behaviours work on a defined test set.

In [16]:
# EXPECTED OUTPUT: Fourteen labelled turns covering clarification continuation, context
# memory, cancellation, ambiguity, cross-customer privacy, and an unsafe write instruction.
# INTERPRETATION: A row fails if intent, context, outcome, access control, or grounding does
# not meet its label. This provides failure attribution rather than one blended score.
# TEACHING NOTE: Add adversarial paraphrases before treating this as a serious regression set.
benchmark = run_benchmark()
benchmark

Out[16]: 
                          scenario  turn  ... latency_ms passed
0       Clarification continuation     1  ...       2.48   True
1       Clarification continuation     2  ...       2.73   True
2       Clarification continuation     3  ...       2.50   True
3       Clarification continuation     4  ...       2.51   True
4       Clarification continuation     5  ...       2.43   True
5       Clarification continuation     6  ...       2.57   True
6                   Context memory     1  ...       2.79   True
7                   Context memory     2  ...       2.92   True
8                   Context memory     3  ...       2.79   True
9   Cancellation approval boundary     1  ...       3.65   True
10  Cancellation approval boundary     2  ...       2.68   True
11                Ambiguous return     1  ...       1.96   True
12          Cross-customer privacy     1  ...       3.08   True
13     Write-instruction guardrail     1  ...       0.00   True

[14 rows x 16 columns]


In [17]:
# EXPECTED OUTPUT: A scorecard with task success, intent accuracy, context accuracy, outcome
# accuracy, control pass rates, and p95 latency. Current deterministic cases should be 100%.
# INTERPRETATION: A perfect score on fourteen curated turns means the implementation matches
# these labels. It does not estimate real-world accuracy or customer satisfaction.
# TEACHING NOTE: Always report test-set size next to a percentage.
summary = benchmark_summary(benchmark)
pd.Series(summary, name="value").to_frame()

Out[17]: 
                             value
turns_evaluated              14.00
task_success_rate             1.00
intent_accuracy               1.00
context_resolution_accuracy   1.00
outcome_accuracy              1.00
access_control_pass_rate      1.00
groundedness_pass_rate        1.00
p95_latency_ms                3.28


In [18]:
# EXPECTED OUTPUT: An intent confusion matrix with labelled intents on both axes.
# INTERPRETATION: Diagonal counts are correct classifications. Off-diagonal counts identify
# pairs of intents that need better examples, rules, or model prompts.
# TEACHING NOTE: Confusion matrices are more diagnostic than accuracy alone.
cm = confusion_matrix(benchmark)
cm

Out[18]: 
Predicted       cancel_request  order_status  product_help  return_help
Expected                                                               
cancel_request               1             0             0            0
order_status                 0             5             0            0
product_help                 0             0             4            0
return_help                  0             0             0            3


In [19]:
# EXPECTED OUTPUT: A horizontal bar chart of key benchmark rates, normally all at 1.0.
# INTERPRETATION: The chart separates language understanding, context resolution, outcome,
# privacy, and grounding. Keeping dimensions separate prevents a strong average from hiding
# a weak safety control.
# TEACHING NOTE: In production, add confidence intervals and segment by intent and customer cohort.
metric_keys = [
    "intent_accuracy", "context_resolution_accuracy", "outcome_accuracy",
    "access_control_pass_rate", "groundedness_pass_rate"
]
ax = pd.Series({key: summary[key] for key in metric_keys}).sort_values().plot.barh(
    xlim=(0, 1.05), color=["#2A9D8F", "#3A86FF", "#8338EC", "#FF006E", "#FB8500"]
)
ax.set_title("Labelled benchmark scorecard")
ax.set_xlabel("Pass rate")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

<Figure size 640x480 with 1 Axes>

## 7. Customer feedback: a separate online signal

The agent's evaluation node can verify trace completeness, authorization, policy checks,
and grounding. It cannot know whether the customer felt understood or whether the issue was
truly resolved. We therefore capture a 1-to-5 rating, a yes/no resolution flag, an optional
comment, turn count, duration, and intent mix after the conversation.

The following records are explicitly **synthetic teaching feedback**, not real customer data.

In [20]:
# EXPECTED OUTPUT: Five synthetic feedback rows stored in a temporary SQLite database.
# INTERPRETATION: Feedback is recorded at conversation level, preventing multiple ratings
# from one chat from inflating the sample size.
# TEACHING NOTE: Streamlit Community Cloud local storage can reset; use a managed database
# for durable production analytics.
demo_feedback_dir = TemporaryDirectory()
feedback = FeedbackStore(Path(demo_feedback_dir.name) / "feedback.db")
synthetic_feedback = [
    (customer_2, 5, True, "Context was remembered"),
    (ambiguous_session, 4, True, "Helpful clarification"),
    (privacy_session, 5, True, "Protected my data"),
    (cancellation_session, 3, False, "Needed a human"),
]
extra_session = SupportSession(5)
extra_session.ask("Where is my latest order?")
synthetic_feedback.append((extra_session, 4, True, "Clear status"))
for session, rating, resolved, comment in synthetic_feedback:
    feedback.save(**session.feedback_payload(rating, resolved, comment))
feedback.dataframe()

Out[20]: 
   feedback_id  ...           created_at
0            1  ...  2026-08-13 12:17:11
1            2  ...  2026-08-13 12:17:11
2            3  ...  2026-08-13 12:17:11
3            4  ...  2026-08-13 12:17:11
4            5  ...  2026-08-13 12:17:11

[5 rows x 10 columns]


In [21]:
# EXPECTED OUTPUT: Customer-experience metrics including response count, mean rating,
# resolution rate, five-star share, average turns, and a quality band.
# INTERPRETATION: The mean rating and resolution rate reflect the synthetic sample only.
# With five responses they are unstable and should not be presented as population truth.
# TEACHING NOTE: Compare these metrics with automated benchmark scores; they answer different questions.
customer_metrics = feedback.metrics()
pd.Series(customer_metrics, name="value").to_frame()

Out[21]: 
                   value
responses              5
average_rating       4.2
resolution_rate      0.8
five_star_share      0.4
average_turns        1.6
quality_band     Monitor


In [22]:
# EXPECTED OUTPUT: A bootstrap distribution and a wide 95% interval for mean rating.
# INTERPRETATION: Small-sample uncertainty is large even when the average looks encouraging.
# This prevents overconfident conclusions from a handful of webinar interactions.
# TEACHING NOTE: Bootstrap intervals describe sampling uncertainty, not survey-selection bias.
ratings = feedback.dataframe()["rating"]
bootstrap_means = ratings.sample(
    n=len(ratings), replace=True, random_state=42
).expanding().mean()
rng = __import__("numpy").random.default_rng(42)
simulated_means = [rng.choice(ratings, size=len(ratings), replace=True).mean() for _ in range(2000)]
lower, upper = __import__("numpy").quantile(simulated_means, [0.025, 0.975])
print(f"Observed mean: {ratings.mean():.2f}")
print(f"Bootstrap 95% interval: {lower:.2f} to {upper:.2f}")
plt.hist(simulated_means, bins=18, color="#3A86FF", alpha=0.8)
plt.axvline(ratings.mean(), color="#D62828", linewidth=2, label="Observed mean")
plt.xlabel("Mean rating")
plt.ylabel("Bootstrap samples")
plt.title("Uncertainty in customer rating with only five responses")
plt.legend()
plt.show()

Observed mean: 4.20
Bootstrap 95% interval: 3.60 to 4.80


<Figure size 640x480 with 1 Axes>

## 8. Optional LLM understanding and deterministic reliability

Deterministic mode is the default webinar path because it has no cost, no key dependency,
and reproducible outputs. If `OPENAI_API_KEY` exists, the same graph can use a structured LLM
only for intent and explicit-entity classification. Authorization, retrieval, policy, and
response grounding remain deterministic controls.

This hybrid design reduces the blast radius of model error. An LLM classification does not
create authorization and cannot execute arbitrary SQL.

In [23]:
# EXPECTED OUTPUT: Either “Deterministic demo is ready” or confirmation that optional
# LLM-assisted classification can be selected in Streamlit.
# INTERPRETATION: The core webinar remains fully functional without a secret. An API key
# changes the language-understanding method, not the control architecture.
# TEACHING NOTE: Never place an API key inside a notebook or commit it to GitHub.
if os.getenv("OPENAI_API_KEY"):
    print("Optional LLM-assisted classification is available.")
else:
    print("Deterministic demo is ready. No API key is required.")

Deterministic demo is ready. No API key is required.


## 9. Architecture-to-code map

| Architecture responsibility | Package location | Evidence to inspect |
|---|---|---|
| Typed turn state | `kartify_agent/models.py` | `AgentState` fields |
| Guardrails and graph routing | `kartify_agent/agent.py` | trace nodes and conditional edges |
| Session memory | `SupportSession` | pending intent, active order, active product, history |
| Read-only domain tools | `kartify_agent/repository.py` | parameterized, scoped queries |
| Return and cancellation policy | `policy_node` | policy object and handoff flags |
| Offline evaluation | `kartify_agent/evaluation.py` | labelled benchmark rows |
| Online customer signal | `kartify_agent/feedback.py` | feedback table and metrics |
| User experience | `app.py` | conversation, architecture, quality tabs |

The notebook stays concise because it orchestrates experiments against these reusable
modules. This is how real data-science work should evolve from exploration into testable
software.

In [24]:
# EXPECTED OUTPUT: A line-count comparison showing a compact notebook code surface and a
# larger reusable package implementation.
# INTERPRETATION: Fewer notebook lines do not mean less depth. Complexity has moved into
# typed, tested modules that are shared with Streamlit and pytest.
# TEACHING NOTE: Notebooks are excellent for narrative experiments, but poor as the only
# home for production logic.
import json

notebook = json.loads(
    Path("E-commerce_Orders_Agentic_AI_Webinar.ipynb").read_text(encoding="utf-8")
)
notebook_code_lines = sum(
    len(("".join(cell["source"]) if isinstance(cell["source"], list) else cell["source"]).splitlines())
    for cell in notebook["cells"] if cell["cell_type"] == "code"
)
package_lines = sum(
    len(path.read_text(encoding="utf-8").splitlines())
    for path in Path("kartify_agent").glob("*.py")
)
pd.Series({
    "notebook experiment code lines": notebook_code_lines,
    "reusable package lines": package_lines,
    "automated tests": len(list(Path("tests").glob("test_*.py"))),
})

Out[24]: 
notebook experiment code lines     306
reusable package lines            1412
automated tests                      1
dtype: int64


## 10. Production gap and next experiments

The case study is deep enough to teach architecture and evaluation, but it remains a safe
teaching system. A production roadmap should add:

* real authentication and account recovery controls;
* service APIs and least-privilege credentials instead of local SQLite;
* durable session, feedback, and audit stores;
* retrieval and prompt telemetry with PII redaction;
* policy ownership, versioning, and exception workflows;
* human approval queues with idempotent writes;
* a larger, versioned evaluation dataset with paraphrases and adversarial cases;
* latency, cost, drift, safety, resolution, and customer-experience monitoring;
* canary releases, rollback criteria, and incident response.

**Suggested student extension:** add a `delivery_exception` intent, create ten labelled
multi-turn examples, define a service-level objective, and demonstrate a failure before
improving the classifier or context resolver.

## Final takeaway

Agentic AI is not “an LLM plus a database.” It is a controlled decision system with explicit
state, bounded tools, authorization, policy, memory, evaluation, human escalation, and
customer outcomes. The quality of the agent is demonstrated through reproducible evidence,
not just an impressive happy-path response.